# 006 - Level6

## 6. Subqueries & CTEs

### 🧠 Técnicas para consultas complejas

- **Subqueries**  
  Permiten utilizar una **consulta dentro de otra** para resolver problemas más complejos.

- **WITH (CTE)**  
  Permite dividir consultas complejas en **pasos más legibles y reutilizables** mediante **Common Table Expressions**.

In [1]:
import pandas as pd
import numpy as np
import polars as pl

In [2]:
df_matches = pd.read_csv('../Data/WorldCupMatches.csv').rename(columns=lambda col: col.lower())
df_players = pd.read_csv('../Data/WorldCupPlayers.csv').rename(columns=lambda col: col.lower())
df_cups = pd.read_csv('../Data/WorldCups.csv').rename(columns=lambda col: col.lower())

pl_matches = pl.read_csv('../Data/WorldCupMatches.csv').rename(lambda col: col.lower())
pl_players = pl.read_csv('../Data/WorldCupPlayers.csv').rename(lambda col: col.lower())
pl_cups = pl.read_csv('../Data/WorldCups.csv').rename(lambda col: col.lower())

## 🛠️ Ejercicio 6.1: El Mundial de los Goleadores

### 🎯 El problema

Identificar el **mundial (o mundiales)** donde se marcaron **más goles** que el **promedio histórico** de todos los mundiales.

---

### 🧠 El reto

No es posible escribir directamente algo como:

```sql
WHERE GoalsScored > AVG(GoalsScored)
```
Esto se debe a que **SQL no permite usar funciones de agregación directamente en la cláusula `WHERE`**.

Para resolver este problema, es necesario utilizar:

- Una **subconsulta**, o
- Una **CTE (`WITH`)**

Ambos enfoques permiten calcular el **promedio histórico** y luego compararlo con **cada mundial** de forma correcta.


Opción CTE
```SQL
WITH consulta_promedio AS(
    SELECT
        year,
        country,
        goalsscored,
        AVG(goalsscored) OVER() AS avg_goals
    FROM worldcup.cups
) SELECT
      *
FROM consulta_promedio
WHERE goalsscored > avg_goals;
```

Opción Subquery
```SQL
SELECT
    year,
    country,
    goalsscored
FROM worldcup.cups
WHERE goalsscored > (SELECT AVG(goalsscored) FROM worldcup.cups);
```

In [36]:
# PANDAS Subquery
df_c = df_cups.copy()

res = df_c[
    (df_c['goalsscored'] > df_c['goalsscored'].mean())
]

res_f = res[['year', 'country', 'goalsscored']]

In [38]:
# PANDAS CTE
df_c = df_cups.copy()

promedio_historico = df_c['goalsscored'].mean()

#df_filtrado = df_c.query(f"goalsscored > {promedio_historico}")
df_filtrado = df_c[df_c['goalsscored'] > promedio_historico]

res = df_filtrado[['year','country','goalsscored']]

In [39]:
# PANDAS Window Functions
df_c = df_c.copy()

# 1. Aseguramos que sea numérico
df_c['goalsscored'] = pd.to_numeric(df_c['goalsscored'], errors='coerce')

# 2. Calculamos el promedio (Esto nos da un solo número, ej: 118.5)
# Esto es como nuestra "Subquery" interna
media_global = df_c['goalsscored'].mean()

# 3. Creamos la columna 'avg_goals' asignando ese número único
# Pandas automáticamente lo repite en todas las filas
df_c['avg_goals'] = media_global

# 4. Filtramos (Nuestra Query Principal)
res = df_c[df_c['goalsscored'] > df_c['avg_goals']]

# 5. Mostramos el resultado
resultado_final = res[['year', 'country', 'goalsscored', 'avg_goals']]

In [40]:
# POLARS Subquery
res = pl_cups.filter(
    pl.col('goalsscored') > pl.col('goalsscored').mean()
).select([
    'year', 'country', 'goalsscored'
])

In [41]:
# POLARS CTE
media = pl_cups.select(pl.col('goalsscored').mean()).item()

res = pl_cups.filter(pl.col('goalsscored') > media).select([
    'year', 'country', 'goalsscored'
])

In [42]:
# POLARS Window Functions
res = pl_cups.with_columns([
    pl.col('goalsscored').mean().alias('avg_goals')
]).filter(
    pl.col('goalsscored') > pl.col('avg_goals')
).select([
    'year','country', 'goalsscored', 'avg_goals'
])

## 🛠️ Ejercicio 6.2: Filtrado Cruzado (Subqueries con IN)

### 🎯 El problema

Obtener la **lista de estadios** (desde la tabla `matches`) donde se jugaron partidos, **pero solo** de aquellos **mundiales** que tuvieron una **asistencia total superior a 2 millones de personas**.

El dato de **asistencia total** se encuentra en la tabla `cups`, por lo que es necesario realizar un **filtrado cruzado** entre ambas tablas.


```SQL
SELECT
    year,
    stadium
FROM worldcup.matches
WHERE year IN (SELECT year
               FROM worldcup.cups
               WHERE (REPLACE(attendance, '.', '')::INTEGER) > 2000000);
```

In [48]:
df_c = df_cups.copy()
df_m = df_matches.copy()

df_c['attendance'] = df_c['attendance'].str.replace('.', '', regex=False).astype(int)

year_list = df_c[df_c['attendance'] > 2000000]['year']

res = df_m[df_m['year'].isin(year_list)]

res_f = res[['year','stadium']].drop_duplicates()

In [53]:
años_con_mucha_gente = (
    pl_cups.filter(
        pl.col('attendance').str.replace_all(r'\.','').cast(pl.Int64, strict=False) > 2000000
    )
    .select('year')
)

res = (
    pl_matches.join(años_con_mucha_gente, on='year', how='semi')
    .select(['year','stadium'])
    .unique()
)

res

year,stadium
i64,str
1990,"""Renato Dall Ara"""
2002,"""Kobe Wing Stadium"""
1986,"""Estadio Ol�mpico Universitario"""
2014,"""Arena Fonte Nova"""
2010,"""Mbombela Stadium"""
…,…
1998,"""Stade du Parc Lescure"""
2006,"""FIFA World Cup Stadium, Munich"""
1982,"""Sarria"""
